# Box Box Box — Parameter Fitter
Reverse-engineer the lap time formula from historical race data.

**Formula:**
```
lap_time = base_lap_time
         + compound_offset[c]
         + deg_rate[c] * max(0, tire_age - cliff[c]) * (1 + temp_scale * (track_temp - 30))
```

**9 parameters:** `soft_offset`, `med_offset`, `soft_deg`, `med_deg`, `hard_deg`, `soft_cliff`, `med_cliff`, `hard_cliff`, `temp_scale`

## 1. Imports & Config

In [7]:
import json
import time
from pathlib import Path
from collections import defaultdict

import numpy as np
from scipy.optimize import minimize, differential_evolution

DATA_DIR = '../data/historical_races'
OUTPUT   = 'fitted_params.json'
TEMP_REF = 30.0  # Normalise temperature around 30°C

# ── Parameters ────────────────────────────────────────────────────────────────
#   p[0] = soft_offset   (< 0, SOFT is faster)
#   p[1] = med_offset    (< 0, MEDIUM is faster)
#   p[2] = soft_deg_rate (> 0)
#   p[3] = med_deg_rate  (> 0)
#   p[4] = hard_deg_rate (> 0)
#   p[5] = soft_cliff    (grace laps before degradation starts)
#   p[6] = med_cliff
#   p[7] = hard_cliff
#   p[8] = temp_scale    (higher temp → more degradation)
PARAM_NAMES = ['soft_offset','med_offset','soft_deg','med_deg','hard_deg',
               'soft_cliff','med_cliff','hard_cliff','temp_scale']
INITIAL     = [-1.5, -0.7, 0.08, 0.04, 0.02, 5.0, 12.0, 20.0, 0.015]
BOUNDS      = [(-5.0, -0.01), (-4.0, -0.01),
               (0.001,  0.6), (0.001,  0.4), (0.001, 0.3),
               (0.0,  30.0), (0.0,  40.0), (0.0,  60.0),
               (-0.05, 0.1)]

# ── Optimiser tuning ──────────────────────────────────────────────────────────
# popsize=15 → population of 15*9=135 candidates (was 20*9=180)
# maxiter=150 → max 150 generations (was 300)
# workers=-1  → use all CPU cores via multiprocessing
# updating='deferred' → required for workers > 1
DE_POPSIZE  = 15
DE_MAXITER  = 150
NUM_FIT     = 500   # races used for fitting
NUM_VAL     = 200   # races used for validation

print('Config loaded ✓')

Config loaded ✓


## 2. Load & Pre-compute Race Data

In [8]:
from tqdm.auto import tqdm
import time

def load_races(n=1000):
    races = []
    for rf in sorted(Path(DATA_DIR).glob('*.json')):
        with open(rf) as f:
            races.extend(json.load(f))
        if len(races) >= n:
            break
    return races[:n]


def precompute(race):
    cfg   = race['race_config']
    base  = cfg['base_lap_time']
    laps  = cfg['total_laps']
    pit_t = cfg['pit_lane_time']
    temp  = cfg['track_temp']

    driver_laps = {}
    fixed_base  = {}

    for strat in race['strategies'].values():
        did  = strat['driver_id']
        tire = strat['starting_tire']
        pits = {ps['lap']: ps['to_tire'] for ps in strat.get('pit_stops', [])}

        lap_seq  = []
        age      = 0
        pit_cost = base * laps
        for lap in range(1, laps + 1):
            age += 1
            lap_seq.append((tire, age))
            if lap in pits:
                pit_cost += pit_t
                tire = pits[lap]
                age  = 0

        driver_laps[did] = lap_seq
        fixed_base[did]  = pit_cost

    return (driver_laps, fixed_base, race['finishing_positions'], temp)


t0 = time.time()
all_races = load_races(NUM_FIT + NUM_VAL)
fit_races = all_races[:NUM_FIT]
val_races = all_races[NUM_FIT:NUM_FIT + NUM_VAL]

precomputed = [
    precompute(r)
    for r in tqdm(fit_races, desc='Pre-computing races', unit='race')
]

print(f'Loaded {len(fit_races)} fit races + {len(val_races)} val races')
print(f'Pre-computed {len(precomputed)} race tensors  ({time.time()-t0:.2f}s)')


Pre-computing races: 100%|██████████| 500/500 [00:00<00:00, 5758.97race/s]

Loaded 500 fit races + 200 val races
Pre-computed 500 race tensors  (0.19s)


## 3. Loss Function

In [9]:
def fast_loss(p, precomputed_races=None):
    """
    Pairwise adjacent-pair ranking loss.
    Penalises whenever a higher-ranked driver is predicted to be slower
    than the driver immediately behind them.
    """
    if precomputed_races is None:
        precomputed_races = precomputed

    offsets  = {'SOFT': p[0], 'MEDIUM': p[1], 'HARD': 0.0}
    deg_rate = {'SOFT': p[2], 'MEDIUM': p[3], 'HARD': p[4]}
    cliff    = {'SOFT': p[5], 'MEDIUM': p[6], 'HARD': p[7]}
    temp_s   = p[8]

    total = 0.0
    for (driver_laps, fixed_base, expected, temp) in precomputed_races:
        temp_mult = 1.0 + temp_s * (temp - TEMP_REF)

        times = {}
        for did, lap_seq in driver_laps.items():
            t = fixed_base[did]
            for (c, age) in lap_seq:
                eff = max(0.0, age - cliff[c])
                t  += offsets[c] + deg_rate[c] * eff * temp_mult
            times[did] = t

        margin = 0.001
        for i in range(len(expected) - 1):
            a, b = expected[i], expected[i + 1]
            diff = times[a] - times[b]  # want diff < 0
            if diff >= -margin:
                total += (diff + margin) ** 2

    return total


# Quick sanity check with initial params
loss0 = fast_loss(np.array(INITIAL))
print(f'Loss at initial params: {loss0:.4f}')
print('Loss function OK ✓')

Loss at initial params: 509976.2066
Loss function OK ✓


## 4. Phase 1 — Global Search (Differential Evolution)

**Why DE?** The loss surface has multiple local minima. DE explores globally first.

**Progress:** The callback prints every generation. Each generation = one full population evaluation.

> **Tip:** If this is still slow, reduce `DE_MAXITER` or `NUM_FIT` in cell 1.

In [10]:
from tqdm.auto import tqdm
import time

_de_pbar  = tqdm(total=DE_MAXITER, desc='DE generations', unit='gen')
_de_log   = []
_de_start = [time.time()]


def de_callback(xk, convergence):
    loss = fast_loss(xk)
    elapsed = time.time() - _de_start[0]
    _de_log.append((len(_de_log) + 1, loss))
    _de_pbar.set_postfix(loss=f'{loss:.6f}', conv=f'{convergence:.4f}', elapsed=f'{elapsed:.1f}s')
    _de_pbar.update(1)


print(f'Starting DE  (popsize={DE_POPSIZE}, maxiter={DE_MAXITER})...')
print(f'Population size = {DE_POPSIZE} × {len(BOUNDS)} params = {DE_POPSIZE * len(BOUNDS)} candidates/gen\n')

_de_start[0] = time.time()
de_result = differential_evolution(
    fast_loss,
    BOUNDS,
    args       = (precomputed,),
    maxiter    = DE_MAXITER,
    tol        = 1e-10,
    seed       = 42,
    popsize    = DE_POPSIZE,
    mutation   = (0.5, 1.5),
    recombination = 0.7,
    workers    = 1,
    updating   = 'immediate',
    init       = 'latinhypercube',
    callback   = de_callback,
    disp       = False,
)
_de_pbar.close()

de_elapsed = time.time() - _de_start[0]
print(f'\nDE finished in {de_elapsed:.1f}s')
print(f'Final loss:  {de_result.fun:.8f}')
print(f'Converged:   {de_result.success}  ({de_result.message})')
print(f'Evaluations: {de_result.nfev}')

print('\n--- DE Best Parameters ---')
for name, val in zip(PARAM_NAMES, de_result.x):
    print(f'  {name:<15} = {val:+.6f}')


Starting DE  (popsize=15, maxiter=150)...
Population size = 15 × 9 params = 135 candidates/gen



KeyboardInterrupt: 

Exception ignored in: <function tqdm.__del__ at 0x000002055923DBC0>
Traceback (most recent call last):
  File "d:\Self_Projects\Github\box-box-box\.venv\Lib\site-packages\tqdm\std.py", line 1148, in __del__
    self.close()
  File "d:\Self_Projects\Github\box-box-box\.venv\Lib\site-packages\tqdm\notebook.py", line 277, in close
    self.disp(bar_style='danger', check_delay=False)
    ^^^^^^^^^
AttributeError: 'tqdm_notebook' object has no attribute 'disp'


### 4a. DE Convergence Plot

In [ ]:
import matplotlib.pyplot as plt

gens, losses = zip(*_de_log)
plt.figure(figsize=(10, 4))
plt.plot(gens, losses, marker='o', markersize=3, linewidth=1.5, color='steelblue')
plt.yscale('log')
plt.xlabel('Generation')
plt.ylabel('Loss (log scale)')
plt.title('Differential Evolution — Convergence')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Phase 2 — Local Refinement (L-BFGS-B)

In [ ]:
from tqdm.auto import tqdm
import time

_lbfgs_pbar  = tqdm(total=5000, desc='L-BFGS-B iters', unit='iter')
_lbfgs_log   = []
_lbfgs_start = time.time()


def lbfgs_callback(xk):
    loss = fast_loss(xk)
    _lbfgs_log.append((len(_lbfgs_log) + 1, loss))
    _lbfgs_pbar.set_postfix(loss=f'{loss:.8f}')
    _lbfgs_pbar.update(1)


print('Starting L-BFGS-B refinement from DE result...')
local_result = minimize(
    fast_loss,
    de_result.x,
    args     = (precomputed,),
    method   = 'L-BFGS-B',
    bounds   = BOUNDS,
    callback = lbfgs_callback,
    options  = {'maxiter': 5000, 'ftol': 1e-20, 'gtol': 1e-12},
)
_lbfgs_pbar.close()

best = local_result.x
print(f'\nL-BFGS-B finished ({time.time() - _lbfgs_start:.1f}s)')
print(f'Final loss: {local_result.fun:.10f}')
print(f'Converged:  {local_result.success}')

print('\n--- Refined Parameters ---')
for name, val in zip(PARAM_NAMES, best):
    print(f'  {name:<15} = {val:+.6f}')


## 6. Validation

In [ ]:
from tqdm.auto import tqdm

def simulate_race(cfg, strategies, p):
    base  = cfg['base_lap_time']
    laps  = cfg['total_laps']
    pit_t = cfg['pit_lane_time']
    temp  = cfg['track_temp']

    offsets  = {'SOFT': p[0], 'MEDIUM': p[1], 'HARD': 0.0}
    deg_rate = {'SOFT': p[2], 'MEDIUM': p[3], 'HARD': p[4]}
    cliff    = {'SOFT': p[5], 'MEDIUM': p[6], 'HARD': p[7]}
    temp_s   = p[8]

    times = {}
    for strat in strategies.values():
        did  = strat['driver_id']
        tire = strat['starting_tire']
        pits = {ps['lap']: ps['to_tire'] for ps in strat.get('pit_stops', [])}

        t = 0.0; age = 0
        for lap in range(1, laps + 1):
            age += 1
            temp_mult = 1.0 + temp_s * (temp - TEMP_REF)
            eff  = max(0.0, age - cliff[tire])
            t   += base + offsets[tire] + deg_rate[tire] * eff * temp_mult
            if lap in pits:
                t   += pit_t
                tire = pits[lap]
                age  = 0
        times[did] = t
    return times


def accuracy(p, races, label=''):
    correct = 0
    for race in tqdm(races, desc=label, unit='race', leave=False):
        times = simulate_race(race['race_config'], race['strategies'], p)
        predicted = sorted(times, key=lambda d: times[d])
        if predicted == race['finishing_positions']:
            correct += 1
    acc = correct / len(races)
    print(f'  {label:<30} {correct}/{len(races)}  ({acc*100:.1f}%)')
    return acc


print('Validation results:')
fit_acc = accuracy(best, fit_races[:200], label='Training (first 200 races)')
val_acc = accuracy(best, val_races,       label=f'Validation ({len(val_races)} races)')

if val_acc < 0.7:
    print('\n⚠️  Accuracy < 70%. Tips:')
    print('   - Increase NUM_FIT or DE_MAXITER in cell 1')


## 7. Save Parameters

In [ ]:
param_dict = dict(zip(PARAM_NAMES, best.tolist()))

with open(OUTPUT, 'w') as f:
    json.dump(param_dict, f, indent=2)

print(f'✅  Saved to {OUTPUT}')
print('\nFinal parameters:')
for k, v in param_dict.items():
    print(f'  {k:<15} = {v:+.6f}')

## 8. Parameter Visualisation

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Plot 1: Degradation curves for each compound ──────────────────────────────
ax = axes[0]
max_laps = 70
laps_range = np.arange(1, max_laps + 1)

compounds = [
    ('SOFT',   best[0], best[2], best[5], 'red'),
    ('MEDIUM', best[1], best[3], best[6], 'gold'),
    ('HARD',   0.0,     best[4], best[7], 'silver'),
]

for name, offset, deg, cliff, color in compounds:
    deg_contrib = [deg * max(0.0, age - cliff) for age in laps_range]
    ax.plot(laps_range, [offset + d for d in deg_contrib],
            label=name, color=color, linewidth=2)

ax.set_xlabel('Tire Age (laps)')
ax.set_ylabel('Lap time delta vs HARD at age 0 (s)')
ax.set_title('Compound Degradation Curves (at TEMP_REF=30°C)')
ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)
ax.legend()
ax.grid(True, alpha=0.3)

# ── Plot 2: Bar chart of all parameters ───────────────────────────────────────
ax2 = axes[1]
colors = ['#e74c3c' if v < 0 else '#2ecc71' for v in best]
bars = ax2.barh(PARAM_NAMES, best, color=colors, edgecolor='white')
ax2.axvline(0, color='black', linewidth=0.8)
ax2.set_title('Fitted Parameter Values')
ax2.set_xlabel('Value')
for bar, val in zip(bars, best):
    ax2.text(bar.get_width() + (0.01 if val >= 0 else -0.01),
             bar.get_y() + bar.get_height() / 2,
             f'{val:+.4f}', va='center',
             ha='left' if val >= 0 else 'right', fontsize=8)
ax2.grid(True, alpha=0.2, axis='x')

plt.tight_layout()
plt.show()